In [2]:
import pandas as pd
import numpy as np

In [38]:
#importing the data, viewing
#some data has '?' instead of null, treating it as a null as well
data = pd.read_csv('fraud_insurance_claims.csv')
data

,months_as_customer,age,policy_number,policy_bind_date,policy_state,policy_csl,policy_deductable,policy_annual_premium,umbrella_limit,insured_zip,...,witnesses,police_report_available,total_claim_amount,injury_claim,property_claim,vehicle_claim,auto_make,auto_model,auto_year,fraud_reported
0,328,48,521585,2014-10-17 00:00:00,OH,250/500,1000,1406.91,0,466132,...,2,YES,71610,6510,13020,52080,Saab,92x,2004,Y
1,228,42,342868,2006-06-27 00:00:00,IN,250/500,2000,1197.22,5000000,468176,...,0,?,5070,780,780,3510,Mercedes,E400,2007,Y
2,134,29,687698,2000-09-06 00:00:00,OH,100/300,2000,1413.14,5000000,430632,...,3,NO,34650,7700,3850,23100,Dodge,RAM,2007,N
3,256,41,227811,1990-05-25 00:00:00,IL,250/500,2000,1415.74,6000000,608117,...,2,NO,63400,6340,6340,50720,Chevrolet,Tahoe,2014,Y
4,228,44,367455,2014-06-06 00:00:00,IL,500/1000,1000,1583.91,6000000,610706,...,1,NO,6500,1300,650,4550,Accura,RSX,2009,N
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,3,38,941851,1991-07-16 00:00:00,OH,500/1000,1000,1310.80,0,431289,...,1,?,87200,17440,8720,61040,Honda,Accord,2006,N
996,285,41,186934,2014-01-05 00:00:00,IL,100/300,1000,1436.79,0,608177,...,3,?,108480,18080,18080,72320,Volkswagen,Passat,2015,N
997,130,34,918516,2003-02-17 00:00:00,OH,250/500,500,1383.49,3000000,442797,...,3,YES,67500,7500,7500,52500,Suburu,Impreza,1996,N
998,458,62,533940,2011-11-18 00:00:00,IL,500/1000,2000,1356.92,5000000,441714,...,1,YES,46980,5220,5220,36540,Audi,A5,1998,N


In [ ]:
#What features am I working with
data.columns

Index(['months_as_customer', 'age', 'policy_number', 'policy_bind_date',
       'policy_state', 'policy_csl', 'policy_deductable',
       'policy_annual_premium', 'umbrella_limit', 'insured_zip', 'insured_sex',
       'insured_education_level', 'insured_occupation', 'insured_hobbies',
       'insured_relationship', 'capital-gains', 'capital-loss',
       'incident_type', 'collision_type', 'incident_severity',
       'authorities_contacted', 'incident_state', 'incident_city',
       'incident_location', 'incident_hour_of_the_day',
       'number_of_vehicles_involved', 'property_damage', 'bodily_injuries',
       'witnesses', 'police_report_available', 'total_claim_amount',
       'injury_claim', 'property_claim', 'vehicle_claim', 'auto_make',
       'auto_model', 'auto_year', 'fraud_reported', 'incident_year',
       'incident_month', 'incident_day', 'time_of_day'],
      dtype='object')

In [39]:
data = data.drop(columns=['incident_location', 'insured_zip']) #Too specific, im not doing any mapping

In [ ]:
#Data integrity check
data['total_claim_check'] = (data['injury_claim'] + data['property_claim'] + data['vehicle_claim'])

(data['total_claim_check'] == data['total_claim_amount']).all()\
#Simple lines of code, simple output, but huge meaning
#This tells me that all the claims data is correct and accurate for each record, which is very important for the business

np.True_

In [30]:
#cleaning
data.isna().sum() #authorities contacted : 91
data['authorities_contacted'].value_counts(dropna=False) #Lists type of authority contacted, going to consider null as none/unknown
data['authorities_contacted'] = data['authorities_contacted'].fillna('None/Unknown')

#police_report_available has ?'s
data['police_report_available'].value_counts(dropna=False) #343 ?'s, too much data to lose, going to consider this as unknown data
data['police_report_available'] = data['police_report_available'].replace('?', 'UNKNOWN')

#All nulls and unknowns taken care of with these lines of code

In [31]:
cleaning_summary = {
    'authorities_contacted': {
        '91 NaNs': 'Replaced with "Unknown"'
    },
    'property_damage': {
        '407 "?" values': 'Replaced with "Unknown"'
    }
}

##Engineering features
#This will help to make aspects of the data easier work with in order to answer specific questions


In [32]:
#Makes dates easier to work with if need be, simple code
data['incident_date'] = pd.to_datetime(
    data['incident_date']
)

data['incident_year'] = data['incident_date'].dt.year
data['incident_month'] = data['incident_date'].dt.month_name()
data['incident_day'] = data['incident_date'].dt.day

data = data.drop(columns='incident_date')

In [33]:
#Question to answer: do more incidents happen in the morning? at night? at what time of day is fruad more frequent?
#This engineered time of day feature makes these questions much easier to analyze by putting the data into buckets
data['time_of_day'] = pd.cut(
    data['incident_hour_of_the_day'],
    bins=[0,6,12,18,24],
    labels=['Night','Morning','Afternoon','Evening']
)

In [34]:
data['vehicle_age'] = (data['incident_year'] - data['auto_year'])

In [35]:
data

,months_as_customer,age,policy_number,policy_bind_date,policy_state,policy_csl,policy_deductable,policy_annual_premium,umbrella_limit,insured_zip,...,vehicle_claim,auto_make,auto_model,auto_year,fraud_reported,incident_year,incident_month,incident_day,time_of_day,vehicle_age
0,328,48,521585,2014-10-17 00:00:00,OH,250/500,1000,1406.91,0,466132,...,52080,Saab,92x,2004,Y,2015,January,25,Night,11
1,228,42,342868,2006-06-27 00:00:00,IN,250/500,2000,1197.22,5000000,468176,...,3510,Mercedes,E400,2007,Y,2015,January,21,Morning,8
2,134,29,687698,2000-09-06 00:00:00,OH,100/300,2000,1413.14,5000000,430632,...,23100,Dodge,RAM,2007,N,2015,February,22,Morning,8
3,256,41,227811,1990-05-25 00:00:00,IL,250/500,2000,1415.74,6000000,608117,...,50720,Chevrolet,Tahoe,2014,Y,2015,January,10,Night,1
4,228,44,367455,2014-06-06 00:00:00,IL,500/1000,1000,1583.91,6000000,610706,...,4550,Accura,RSX,2009,N,2015,February,17,Evening,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,3,38,941851,1991-07-16 00:00:00,OH,500/1000,1000,1310.80,0,431289,...,61040,Honda,Accord,2006,N,2015,February,22,Evening,9
996,285,41,186934,2014-01-05 00:00:00,IL,100/300,1000,1436.79,0,608177,...,72320,Volkswagen,Passat,2015,N,2015,January,24,Evening,0
997,130,34,918516,2003-02-17 00:00:00,OH,250/500,500,1383.49,3000000,442797,...,52500,Suburu,Impreza,1996,N,2015,January,23,Night,19
998,458,62,533940,2011-11-18 00:00:00,IL,500/1000,2000,1356.92,5000000,441714,...,36540,Audi,A5,1998,N,2015,February,26,Night,17


In [6]:
data.describe() #describing the data gives highly valuable information such as min, max, mean, etc without the need to calculate manually

,months_as_customer,age,policy_number,policy_deductable,policy_annual_premium,umbrella_limit,insured_zip,capital-gains,capital-loss,incident_hour_of_the_day,number_of_vehicles_involved,bodily_injuries,witnesses,total_claim_amount,injury_claim,property_claim,vehicle_claim,auto_year
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1.000000e+03,1000.000000,1000.000000,1000.000000,1000.000000,1000.00000,1000.000000,1000.000000,1000.00000,1000.000000,1000.000000,1000.000000,1000.000000
mean,203.954000,38.948000,546238.648000,1136.000000,1256.406150,1.101000e+06,501214.488000,25126.100000,-26793.700000,11.644000,1.83900,0.992000,1.487000,52761.94000,7433.420000,7399.570000,37928.950000,2005.103000
std,115.113174,9.140287,257063.005276,611.864673,244.167395,2.297407e+06,71701.610941,27872.187708,28104.096686,6.951373,1.01888,0.820127,1.111335,26401.53319,4880.951853,4824.726179,18886.252893,6.015861
min,0.000000,19.000000,100804.000000,500.000000,433.330000,-1.000000e+06,430104.000000,0.000000,-111100.000000,0.000000,1.00000,0.000000,0.000000,100.00000,0.000000,0.000000,70.000000,1995.000000
25%,115.750000,32.000000,335980.250000,500.000000,1089.607500,0.000000e+00,448404.500000,0.000000,-51500.000000,6.000000,1.00000,0.000000,1.000000,41812.50000,4295.000000,4445.000000,30292.500000,2000.000000
50%,199.500000,38.000000,533135.000000,1000.000000,1257.200000,0.000000e+00,466445.500000,0.000000,-23250.000000,12.000000,1.00000,1.000000,1.000000,58055.00000,6775.000000,6750.000000,42100.000000,2005.000000
75%,276.250000,44.000000,759099.750000,2000.000000,1415.695000,0.000000e+00,603251.000000,51025.000000,0.000000,17.000000,3.00000,2.000000,2.000000,70592.50000,11305.000000,10885.000000,50822.500000,2010.000000
max,479.000000,64.000000,999435.000000,2000.000000,2047.590000,1.000000e+07,620962.000000,100500.000000,0.000000,23.000000,4.00000,2.000000,3.000000,114920.00000,21450.000000,23670.000000,79560.000000,2015.000000


In [7]:
#Min of 100 could be an error, need to investigate
data['total_claim_amount'].describe() 

min_claim = data[data['total_claim_amount']==100]
min_claim

,months_as_customer,age,policy_number,policy_bind_date,policy_state,policy_csl,policy_deductable,policy_annual_premium,umbrella_limit,insured_zip,...,witnesses,police_report_available,total_claim_amount,injury_claim,property_claim,vehicle_claim,auto_make,auto_model,auto_year,fraud_reported
775,89,32,266247,2015-01-17 00:00:00,IN,100/300,2000,1482.53,0,620358,...,2,UNKNOWN,100,10,20,70,Audi,A3,2002,N


In [8]:
#Verifying consistency, all good
data['incident_type'].unique()

array(['Single Vehicle Collision', 'Vehicle Theft',
       'Multi-vehicle Collision', 'Parked Car'], dtype=object)

In [45]:
#Questions to explore, how much fraud? Is there a group with a high fraud rate,
#distribution of claims
